# EXP/0413 Analysis --- Seller Strategy: `tit_for_tat_matcher`

**Fractional model hypothesis**:

$$s_t = v + (\text{market} - v)\,\big[\beta_0 + \beta_1\,\text{frac}(m_t) + \beta_2\,\text{frac}(s_{t-1})\big]$$

where $\text{frac}(x) = (x - v)/(\text{market} - v)$.

This notebook analyzes seller strategy **Tit-for-Tat Matcher** with:
1. **Shape discovery**: bin by $s_{t-1}$, plot $s_t$ vs $m_t$ to reveal piecewise structure
2. **Time homogeneity**: split rounds into 6 chunks, cross-prediction confusion matrix
3. **Covariate ablation**: test adding $v$, multi-step history, initial prices

---

## S0. Setup

In [ ]:
import json, sys, glob, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm
%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.figsize'] = (10, 5)
warnings.filterwarnings('ignore')

STRATEGY = "tit_for_tat_matcher"
STRATEGY_LABEL = "Tit-for-Tat Matcher"
RESULTS_DIR = Path("results") / STRATEGY

# ── OLS helper ──
def ols_fit(X, y):
    n, p = X.shape
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ beta
    resid = y - y_hat
    rmse = float(np.sqrt(np.mean(resid**2)))
    ss_res = float(np.sum(resid**2))
    ss_tot = float(np.sum((y - np.mean(y))**2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 1e-12 else 1.0
    dof = max(n - p, 1)
    s2 = ss_res / dof
    try:
        se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    except:
        se = np.full(p, np.nan)
    return beta, rmse, r2, y_hat, se

# ── Data loader ──
def load_all_data(strategy_dir):
    trans = []
    runs_all = []
    for f in sorted(glob.glob(str(strategy_dir / "*.json"))):
        if "transitions" in f:
            continue
        data = json.load(open(f))
        if "runs" not in data:
            continue
        for run in data["runs"]:
            meta = run["metadata"]
            v = meta["v"]
            cost = meta["product"]["cost"]
            market = meta["product"]["market_price"]
            rounds = run["rounds"]
            m_init = rounds[0]["buyer_price"]
            s_init = rounds[0]["supplier_price"]
            run_idx = len(runs_all)
            runs_all.append(run)
            for i in range(1, len(rounds)):
                prev = rounds[i-1]; curr = rounds[i]
                s_prev = prev["supplier_price"]
                m_curr = curr["buyer_price"]
                s_curr = curr["supplier_price"]
                if s_prev is None or s_curr is None:
                    continue
                trans.append(dict(
                    run_idx=run_idx, round_t=curr["round"],
                    m_t=m_curr, s_prev=s_prev, s_t=s_curr,
                    v=v, cost=cost, market=market,
                    strategy=meta["strategy"],
                    supplier_id=meta["supplier"]["id"],
                    product=meta["product"]["name"],
                    buyer_type=meta["buyer_type"]["name"],
                    template_id=curr.get("template_id", -1),
                    m_init=m_init,
                    s_init=s_init if s_init is not None else np.nan,
                    m_prev=prev["buyer_price"],
                    s_prev2=rounds[i-2]["supplier_price"] if i >= 2 else None,
                ))
    return trans, runs_all

trans, runs_all = load_all_data(RESULTS_DIR)
print(f"Strategy: {STRATEGY_LABEL}")
print(f"Transitions: {len(trans)}")
print(f"Runs: {len(runs_all)}")
print(f"Products: {sorted(set(t['product'] for t in trans))}")
print(f"Buyer types: {sorted(set(t['buyer_type'] for t in trans))}")
print(f"v values: {sorted(set(round(t['v'],2) for t in trans))}")
print(f"Suppliers: {sorted(set(t['supplier_id'] for t in trans))}")

# ── Extract arrays ──
m   = np.array([t["m_t"] for t in trans])
s   = np.array([t["s_prev"] for t in trans])
y   = np.array([t["s_t"] for t in trans])
v   = np.array([t["v"] for t in trans])
mkt = np.array([t["market"] for t in trans])
ti  = np.array([t["round_t"] for t in trans], dtype=float)

# Fractional coordinates
zone = mkt - v
zone[zone < 0.01] = 0.01
fm = (m - v) / zone
fs = (s - v) / zone
fy = (y - v) / zone

print(f"\nm_t range: [{m.min():.2f}, {m.max():.2f}]")
print(f"s_prev range: [{s.min():.2f}, {s.max():.2f}]")
print(f"s_t range: [{y.min():.2f}, {y.max():.2f}]")
print(f"frac(m_t) range: [{fm.min():.2f}, {fm.max():.2f}]")

---
## S1. Shape Discovery: What Does $s_t = f(m_t \mid s_{t-1})$ Look Like?

We bin the data by $s_{t-1}$ quartiles, then plot $s_t$ vs $m_t$ within each bin.
This reveals the functional form:
- **Linear region**: when $m_t$ is within the negotiation zone, $s_t$ responds linearly
- **Floor region**: when $m_t < v$, the seller refuses to go below $v$, creating a flat floor
- **Acceptance region**: when $m_t$ is high enough, the seller simply accepts ($s_t \approx m_t$)

### S1.1 Raw scatter colored by $s_{t-1}$

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(m, y, c=s, cmap='viridis', alpha=0.6, s=20, edgecolors='none')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("$s_{t-1}$ (seller's previous price)", fontsize=10)

# Reference lines
lo = min(m.min(), y.min()) * 0.9
hi = max(m.max(), y.max()) * 1.05
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=0.5, label="$s_t = m_t$ (acceptance)")

# v floor lines for each unique v
for v_val in sorted(set(v)):
    ax.axhline(v_val, color='red', ls=':', lw=1, alpha=0.5)
    ax.text(lo + 0.1, v_val + 0.05, f"v={v_val:.2f}", fontsize=7, color='red')

ax.set_xlabel("Buyer price $m_t$ (\$)", fontsize=11)
ax.set_ylabel("Seller price $s_t$ (\$)", fontsize=11)
ax.set_title(f"Figure 1: Raw scatter $s_t$ vs $m_t$, colored by $s_{{t-1}}$\n"
             f"Strategy: {STRATEGY_LABEL}  |  N={len(m)}", fontsize=12)
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

### S1.2 Binned by $s_{t-1}$ quartiles

Within each $s_{t-1}$ bin, we plot $s_t$ vs $m_t$ and fit a local linear model.
This reveals whether the slope $\partial s_t / \partial m_t$ changes with $s_{t-1}$.

In [ ]:
# Bin by s_prev quartiles
n_bins = 4
s_edges = np.percentile(s, np.linspace(0, 100, n_bins + 1))
s_edges[-1] += 0.01  # include max

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for bi in range(n_bins):
    ax = axes[bi]
    mask = (s >= s_edges[bi]) & (s < s_edges[bi + 1])
    mg, yg, vg = m[mask], y[mask], v[mask]

    ax.scatter(mg, yg, alpha=0.5, s=15, edgecolors='none')

    # Fit local linear
    if mask.sum() >= 5:
        Xg = np.column_stack([np.ones(mask.sum()), mg])
        bg, rg, r2g, ypg, _ = ols_fit(Xg, yg)
        x_range = np.linspace(mg.min(), mg.max(), 100)
        ax.plot(x_range, bg[0] + bg[1] * x_range, 'r-', lw=2,
                label=f"OLS: {bg[0]:.2f} + {bg[1]:.2f}$\\cdot m_t$\nR$^2$={r2g:.3f}")

    # Acceptance line
    lo_ax = min(mg.min(), yg.min()) * 0.9
    hi_ax = max(mg.max(), yg.max()) * 1.05
    ax.plot([lo_ax, hi_ax], [lo_ax, hi_ax], 'k--', lw=0.8, alpha=0.4)

    # v floor
    for v_val in sorted(set(vg)):
        ax.axhline(v_val, color='red', ls=':', lw=0.8, alpha=0.5)

    ax.set_xlabel("$m_t$"); ax.set_ylabel("$s_t$")
    ax.set_title(f"$s_{{t-1}} \\in [{s_edges[bi]:.2f},\\, {s_edges[bi+1]:.2f})$"
                 f"  (N={mask.sum()})", fontsize=10)
    ax.legend(fontsize=7, loc='upper left')

fig.suptitle(f"Figure 2: $s_t$ vs $m_t$, binned by $s_{{t-1}}$ quartiles\n"
             f"Strategy: {STRATEGY_LABEL}", y=1.02, fontsize=12)
fig.tight_layout(); plt.show()

### S1.3 Piecewise structure detection

Test three candidate models and compare in-sample RMSE:
- **Linear**: $s_t = \beta_0 + \beta_1 m_t + \beta_2 s_{t-1}$
- **Piecewise-linear**: separate slopes for $m_t < v$, $v \leq m_t \leq s_{t-1}$, $m_t > s_{t-1}$
- **Fractional linear**: $\text{frac}(s_t) = \beta_0 + \beta_1 \text{frac}(m_t) + \beta_2 \text{frac}(s_{t-1})$
- **Clipped fractional**: same as fractional but $\text{frac}(m_t)$ clipped to $[0, 1]$

In [ ]:
# ── Model candidates ──
def fit_and_report(label, X, y_target, n_params=None):
    if n_params is None:
        n_params = X.shape[1]
    b, rmse, r2, yh, se = ols_fit(X, y_target)
    return {"label": label, "rmse": rmse, "r2": r2, "n_params": n_params,
            "beta": b, "se": se, "y_hat": yh}

results = []

# 1. Raw linear
results.append(fit_and_report("Linear (raw)", np.column_stack([np.ones_like(m), m, s]), y))

# 2. Fractional linear
results.append(fit_and_report("Fractional linear",
    np.column_stack([np.ones_like(fm), fm, fs]), fy))

# 3. Clipped fractional (clip frac(m_t) to [0, 1])
fm_clip = np.clip(fm, 0, 1)
results.append(fit_and_report("Fractional clipped",
    np.column_stack([np.ones_like(fm_clip), fm_clip, fs]), fy))

# 4. Piecewise by regime: separate intercept+slope for each of 3 regimes
regime = np.zeros_like(m, dtype=int)
regime[m < v] = 0       # below v
regime[(m >= v) & (m <= s)] = 1  # negotiation zone
regime[m > s] = 2       # buyer above seller

# Build indicator columns: for each regime, (indicator, indicator*m_t)
# Plus s_prev as a global feature
X_pw_cols = [s]  # s_prev is always a feature
for r_val in [0, 1, 2]:
    ind = (regime == r_val).astype(float)
    X_pw_cols.append(ind)            # regime intercept
    X_pw_cols.append(ind * m)        # regime-specific slope on m
X_pw = np.column_stack(X_pw_cols)
results.append(fit_and_report("Piecewise (3 regimes)", X_pw, y))

# 5. Quadratic
X_q = np.column_stack([np.ones_like(m), m, s, m**2, s**2, m*s])
results.append(fit_and_report("Quadratic", X_q, y))

# Report
print(f"{'Model':<25} {'RMSE':>8} {'R2':>8} {'#Params':>8}")
print("-" * 55)
for r in sorted(results, key=lambda x: x['rmse']):
    print(f"{r['label']:<25} {r['rmse']:>8.4f} {r['r2']:>8.4f} {r['n_params']:>8}")

### S1.4 Regime-specific scatter

**Figure 3.** Scatter colored by regime: $m_t < v$ (below floor), $v \leq m_t \leq s_{t-1}$ (zone), $m_t > s_{t-1}$ (above seller).

In [ ]:
labels = ['$m_t < v$ (below floor)', '$v \\leq m_t \\leq s_{t-1}$ (zone)', '$m_t > s_{t-1}$ (above seller)']
colors_r = ['C3', 'C0', 'C2']
fig, ax = plt.subplots(figsize=(9, 7))
for r_val in [0, 1, 2]:
    mask_r = regime == r_val
    if mask_r.sum() == 0: continue
    ax.scatter(m[mask_r], y[mask_r], alpha=0.5, s=15, c=colors_r[r_val],
              label=f"{labels[r_val]} (N={mask_r.sum()})", edgecolors='none')

lo = min(m.min(), y.min()) * 0.9; hi = max(m.max(), y.max()) * 1.05
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.4, label='$s_t=m_t$')
for v_val in sorted(set(v)):
    ax.axhline(v_val, color='red', ls=':', lw=1, alpha=0.5)
    ax.axvline(v_val, color='red', ls=':', lw=0.5, alpha=0.3)
ax.set_xlabel("$m_t$ (\\$)"); ax.set_ylabel("$s_t$ (\\$)")
ax.set_title(f"Figure 3: Three regimes of seller behavior\nStrategy: {STRATEGY_LABEL}", fontsize=12)
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

# Per-regime stats
print(f"{'Regime':<30} {'N':>5} {'mean s_t':>9} {'std s_t':>9} {'accept%':>8}")
print("-" * 65)
for r_val in [0, 1, 2]:
    mask_r = regime == r_val
    if mask_r.sum() == 0: continue
    accept = np.mean(np.abs(y[mask_r] - m[mask_r]) < 0.05) * 100
    print(f"{labels[r_val]:<30} {mask_r.sum():>5} {y[mask_r].mean():>9.2f} "
          f"{y[mask_r].std():>9.2f} {accept:>7.1f}%")

---
## S2. Time Homogeneity: Does the Model Change Across Rounds?

Split rounds 2--12 into chunks, fit the best model (from S1) on each chunk,
then build a cross-prediction confusion matrix.

Chunks: [2--3], [4--5], [6--7], [8--9], [10--11], [12] (or adaptive if data is sparse).

### S2.1 Per-chunk model fit

In [ ]:
# Use fractional linear as the base model (can swap based on S1 results)
# Define chunks
all_rounds = sorted(set(ti))
n_chunks = min(6, len(all_rounds) // 2 + 1)
chunk_edges = np.array_split(sorted(all_rounds), n_chunks)
chunks = [(int(c[0]), int(c[-1])) for c in chunk_edges]

print(f"Chunks: {chunks}")

chunk_betas = {}
print(f"\n{'Chunk':<15} {'Rounds':<12} {'N':>5} {'b0':>7} {'b(fm)':>7} {'b(fs)':>7} {'RMSE':>7} {'R2':>7}")
print("-" * 75)

for ci, (r_lo, r_hi) in enumerate(chunks):
    mask = (ti >= r_lo) & (ti <= r_hi)
    if mask.sum() < 5:
        print(f"Chunk {ci:<9} [{r_lo}-{r_hi}]{'':>6} {mask.sum():>5}  (too few)")
        continue
    Xc = np.column_stack([np.ones(mask.sum()), fm[mask], fs[mask]])
    bc, rc, r2c, _, _ = ols_fit(Xc, fy[mask])
    chunk_betas[ci] = {"beta": bc, "rounds": (r_lo, r_hi), "n": int(mask.sum())}
    print(f"Chunk {ci:<9} [{r_lo}-{r_hi}]{'':>6} {mask.sum():>5} {bc[0]:>+7.3f} "
          f"{bc[1]:>+7.3f} {bc[2]:>+7.3f} {rc:>7.4f} {r2c:>7.4f}")

### S2.2 Cross-prediction confusion matrix

Cell $(i, j)$ = RMSE when model trained on chunk $i$ predicts chunk $j$.
Diagonal = in-sample. Off-diagonal = generalization.

If the matrix is approximately uniform, the model is **time-homogeneous**.
If diagonal is much smaller than off-diagonal, the model is **step-dependent**.

In [ ]:
valid_chunks = sorted(chunk_betas.keys())
n_valid = len(valid_chunks)

if n_valid >= 2:
    conf_matrix = np.full((n_valid, n_valid), np.nan)

    for i, ci in enumerate(valid_chunks):
        r_lo_i, r_hi_i = chunk_betas[ci]["rounds"]
        mask_i = (ti >= r_lo_i) & (ti <= r_hi_i)
        Xi = np.column_stack([np.ones(mask_i.sum()), fm[mask_i], fs[mask_i]])
        yi = fy[mask_i]
        bi = chunk_betas[ci]["beta"]

        for j, cj in enumerate(valid_chunks):
            r_lo_j, r_hi_j = chunk_betas[cj]["rounds"]
            mask_j = (ti >= r_lo_j) & (ti <= r_hi_j)
            Xj = np.column_stack([np.ones(mask_j.sum()), fm[mask_j], fs[mask_j]])
            yj = fy[mask_j]
            pred_j = Xj @ bi
            conf_matrix[i, j] = float(np.sqrt(np.mean((yj - pred_j)**2)))

    fig, ax = plt.subplots(figsize=(8, 6.5))
    im = ax.imshow(conf_matrix, cmap='YlOrRd', aspect='auto')
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("RMSE (fractional units)")

    chunk_labels = [f"R{chunk_betas[c]['rounds'][0]}-{chunk_betas[c]['rounds'][1]}" for c in valid_chunks]
    ax.set_xticks(range(n_valid))
    ax.set_xticklabels(chunk_labels, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(n_valid))
    ax.set_yticklabels(chunk_labels, fontsize=9)
    ax.set_xlabel("Test chunk (predicted)")
    ax.set_ylabel("Train chunk (model fitted on)")

    for i in range(n_valid):
        for j in range(n_valid):
            ax.text(j, i, f"{conf_matrix[i,j]:.3f}", ha='center', va='center', fontsize=8,
                    color='white' if conf_matrix[i,j] > np.nanmedian(conf_matrix) else 'black')

    ax.set_title(f"Figure 4: Cross-prediction RMSE matrix (fractional model)\\n"
                 f"Strategy: {STRATEGY_LABEL}  |  Diagonal = in-sample", fontsize=11)
    fig.tight_layout(); plt.show()

    # Summary stats
    diag = np.diag(conf_matrix)
    off_diag = conf_matrix[~np.eye(n_valid, dtype=bool)]
    print(f"Diagonal (in-sample) mean RMSE:  {np.nanmean(diag):.4f}")
    print(f"Off-diagonal (cross) mean RMSE:  {np.nanmean(off_diag):.4f}")
    print(f"Ratio (off/diag):                {np.nanmean(off_diag)/np.nanmean(diag):.2f}x")
    print(f"  -> {'TIME-HOMOGENEOUS (ratio < 1.5)' if np.nanmean(off_diag)/np.nanmean(diag) < 1.5 else 'STEP-DEPENDENT (ratio >= 1.5)'}")
else:
    print("Not enough chunks with sufficient data for cross-prediction.")

### S2.3 Coefficient stability across chunks

In [ ]:
if len(valid_chunks) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    coef_names = ['$\\beta_0$', '$\\beta_1$ (frac $m_t$)', '$\\beta_2$ (frac $s_{t-1}$)']
    for ci_idx, cname in enumerate(coef_names):
        ax = axes[ci_idx]
        vals = [chunk_betas[c]["beta"][ci_idx] for c in valid_chunks]
        labels_c = [f"R{chunk_betas[c]['rounds'][0]}-{chunk_betas[c]['rounds'][1]}" for c in valid_chunks]
        ax.bar(range(len(vals)), vals, alpha=0.8)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(labels_c, rotation=30, ha='right', fontsize=8)
        ax.set_title(cname)
        ax.axhline(0, color='gray', lw=0.5)
    fig.suptitle(f"Figure 5: Fractional model coefficients by chunk\\nStrategy: {STRATEGY_LABEL}",
                 y=1.02, fontsize=11)
    fig.tight_layout(); plt.show()

---
## S3. Covariate Ablation: What Features Matter?

### S3.1 Feature hierarchy

Test a hierarchy of models to determine the minimal sufficient feature set.
All models use the **fractional** coordinate system where applicable.

In [ ]:
si_arr = np.array([t["s_init"] for t in trans])
mi_arr = np.array([t["m_init"] for t in trans])
mp_arr = np.array([t["m_prev"] for t in trans])
sp2_raw = [t["s_prev2"] for t in trans]
has_sp2 = np.array([x is not None for x in sp2_raw])
sp2_arr = np.array([x if x is not None else 0.0 for x in sp2_raw])

# Fractional versions of extra features
fi_m_init = (mi_arr - v) / zone  # frac of initial buyer price
fi_s_init = np.where(~np.isnan(si_arr), (si_arr - v) / zone, np.nan)
fi_m_prev = (mp_arr - v) / zone  # frac of previous buyer price
fi_sp2 = np.where(has_sp2, (sp2_arr - v) / zone, 0.0)

valid_si = ~np.isnan(fi_s_init)

ablations = [
    # (label, X, y_target, mask)
    ("(a) frac(m_t) only",
     lambda: (np.column_stack([np.ones_like(fm), fm]), fy, np.ones(len(fm), dtype=bool))),
    ("(b) frac(s_prev) only",
     lambda: (np.column_stack([np.ones_like(fs), fs]), fy, np.ones(len(fs), dtype=bool))),
    ("(c) frac(m_t) + frac(s_prev)  [BASELINE]",
     lambda: (np.column_stack([np.ones_like(fm), fm, fs]), fy, np.ones(len(fm), dtype=bool))),
    ("(d) + v  (valuation)",
     lambda: (np.column_stack([np.ones_like(fm), fm, fs, v]), fy, np.ones(len(fm), dtype=bool))),
    ("(e) + t  (round index)",
     lambda: (np.column_stack([np.ones_like(fm), fm, fs, ti]), fy, np.ones(len(fm), dtype=bool))),
    ("(f) + frac(m_prev)  (2-step buyer)",
     lambda: (np.column_stack([np.ones_like(fm), fm, fs, fi_m_prev]), fy, np.ones(len(fm), dtype=bool))),
    ("(g) + frac(s_prev2)  (2-step seller, rounds>=3)",
     lambda: (np.column_stack([np.ones(has_sp2.sum()), fm[has_sp2], fs[has_sp2], fi_sp2[has_sp2]]),
              fy[has_sp2], has_sp2)),
    ("(h) + frac(m_init)  (initial buyer price)",
     lambda: (np.column_stack([np.ones(valid_si.sum()), fm[valid_si], fs[valid_si], fi_m_init[valid_si]]),
              fy[valid_si], valid_si)),
    ("(i) + frac(s_init)  (initial seller price)",
     lambda: (np.column_stack([np.ones(valid_si.sum()), fm[valid_si], fs[valid_si], fi_s_init[valid_si]]),
              fy[valid_si], valid_si)),
    ("(j) + frac(m_prev) + frac(s_prev2)  (full 2-step, rounds>=3)",
     lambda: (np.column_stack([np.ones(has_sp2.sum()), fm[has_sp2], fs[has_sp2],
                                fi_m_prev[has_sp2], fi_sp2[has_sp2]]),
              fy[has_sp2], has_sp2)),
]

print(f"{'Model':<50} {'N':>5} {'RMSE':>8} {'R2':>8} {'Extra coef':>12}")
print("-" * 90)

for label, builder in ablations:
    X, y_target, mask = builder()
    if len(y_target) < 5:
        print(f"{label:<50} {len(y_target):>5}  (too few)")
        continue
    b, rmse, r2, _, se = ols_fit(X, y_target)
    # Show last coefficient if model has extras
    extra = f"{b[-1]:>+12.4f}" if X.shape[1] > 3 else ""
    print(f"{label:<50} {len(y_target):>5} {rmse:>8.4f} {r2:>8.4f} {extra}")

### S3.2 Baseline model coefficient table

Full OLS output for the fractional baseline: $\text{frac}(s_t) = \beta_0 + \beta_1 \text{frac}(m_t) + \beta_2 \text{frac}(s_{t-1})$

In [ ]:
X_base = np.column_stack([np.ones_like(fm), fm, fs])
b, rmse, r2, _, se = ols_fit(X_base, fy)
names = ["intercept", "frac(m_t)", "frac(s_prev)"]

print(f"Fractional baseline: N={len(fy)}, RMSE={rmse:.4f}, R2={r2:.4f}")
print(f"\n  {'Feature':<16} {'Coef':>9} {'SE':>9} {'t-stat':>9}")
print(f"  {'---'*15}")
for i, name in enumerate(names):
    t_stat = b[i]/se[i] if se[i] > 1e-8 else float('inf')
    sig = "***" if abs(t_stat)>3.29 else "**" if abs(t_stat)>2.58 else "*" if abs(t_stat)>1.96 else ""
    print(f"  {name:<16} {b[i]:>+9.4f} {se[i]:>9.4f} {t_stat:>+9.2f} {sig}")

# Reconstruct in price space
print(f"\nIn price space interpretation:")
print(f"  s_t = v + (market - v) * [{b[0]:.3f} + {b[1]:.3f}*frac(m_t) + {b[2]:.3f}*frac(s_prev)]")
print(f"  The seller places itself at {b[0]*100:.1f}% of the zone by default,")
print(f"  + {b[1]*100:.1f}% * buyer's position + {b[2]*100:.1f}% * own previous position")

### S3.3 Per-product and per-supplier stability

Fit the fractional baseline separately within each product and supplier.
Stable coefficients = the fractional normalization works across products.

In [ ]:
print(f"{'Condition':<30} {'N':>5} {'b0':>7} {'b(fm)':>7} {'b(fs)':>7} {'RMSE':>7} {'R2':>6}")
print("-" * 75)
for key in ["product", "supplier_id", "buyer_type"]:
    groups = {}
    for idx, t in enumerate(trans):
        groups.setdefault(t[key], []).append(idx)
    for name, idxs in sorted(groups.items()):
        if len(idxs) < 5: continue
        mask = np.array(idxs)
        Xg = np.column_stack([np.ones(len(mask)), fm[mask], fs[mask]])
        bg, rg, r2g, _, _ = ols_fit(Xg, fy[mask])
        tag = f"  {name[:28]}"
        print(f"{tag:<30} {len(mask):>5} {bg[0]:>+7.3f} {bg[1]:>+7.3f} {bg[2]:>+7.3f} {rg:>7.4f} {r2g:>6.4f}")
    print()

### S3.4 True vs Predicted: best model

**Figure 6.** Scatter of true vs predicted $\text{frac}(s_t)$ for the fractional baseline model.

In [ ]:
b_best, _, r2_best, yhat_best, _ = ols_fit(np.column_stack([np.ones_like(fm), fm, fs]), fy)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: fractional space
ax = axes[0]
lo = min(fy.min(), yhat_best.min()) * 0.9
hi = max(fy.max(), yhat_best.max()) * 1.1
ax.scatter(fy, yhat_best, alpha=0.4, s=12, edgecolors='none')
ax.plot([lo,hi],[lo,hi],'k--',lw=1)
ax.set_xlabel("True frac($s_t$)"); ax.set_ylabel("Predicted frac($s_t$)")
ax.set_title(f"Fractional space\nRMSE={np.sqrt(np.mean((fy-yhat_best)**2)):.4f}  R2={r2_best:.4f}")
ax.set_aspect('equal', adjustable='box')

# Right: price space
ax = axes[1]
y_pred_price = v + zone * yhat_best
lo = min(y.min(), y_pred_price.min()) * 0.9
hi = max(y.max(), y_pred_price.max()) * 1.05
ax.scatter(y, y_pred_price, alpha=0.4, s=12, edgecolors='none')
ax.plot([lo,hi],[lo,hi],'k--',lw=1)
rmse_price = np.sqrt(np.mean((y - y_pred_price)**2))
ss_r = np.sum((y - y_pred_price)**2)
ss_t = np.sum((y - np.mean(y))**2)
r2_price = 1 - ss_r/ss_t
ax.set_xlabel("True $s_t$ (\\$)"); ax.set_ylabel("Predicted $s_t$ (\\$)")
ax.set_title(f"Price space\nRMSE=\\${rmse_price:.4f}  R2={r2_price:.4f}")
ax.set_aspect('equal', adjustable='box')

fig.suptitle(f"Figure 6: Fractional baseline model fit\\nStrategy: {STRATEGY_LABEL}", y=1.02, fontsize=12)
fig.tight_layout(); plt.show()

---
## S4. Residual Diagnostics

Check for systematic patterns in the fractional model residuals.

In [ ]:
resid = fy - yhat_best

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Residual vs m_t
ax = axes[0, 0]
ax.scatter(fm, resid, alpha=0.4, s=10, edgecolors='none')
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_xlabel("frac($m_t$)"); ax.set_ylabel("Residual")
ax.set_title("Residual vs frac($m_t$)")

# Residual vs s_prev
ax = axes[0, 1]
ax.scatter(fs, resid, alpha=0.4, s=10, edgecolors='none')
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_xlabel("frac($s_{t-1}$)"); ax.set_ylabel("Residual")
ax.set_title("Residual vs frac($s_{t-1}$)")

# Residual vs round
ax = axes[1, 0]
rounds_unique = sorted(set(ti))
means = [resid[ti == r].mean() for r in rounds_unique]
stds = [resid[ti == r].std() for r in rounds_unique]
ax.bar([int(r) for r in rounds_unique], means, yerr=stds, capsize=3, alpha=0.7, edgecolor='black')
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_xlabel("Round $t$"); ax.set_ylabel("Mean residual")
ax.set_title("Residual by round (+-1 SD)")

# Histogram
ax = axes[1, 1]
ax.hist(resid, bins=30, alpha=0.7, edgecolor='black')
ax.axvline(0, color='red', ls='--', lw=1)
ax.set_xlabel("Residual"); ax.set_ylabel("Count")
ax.set_title(f"Residual distribution (mean={resid.mean():.4f}, std={resid.std():.4f})")

fig.suptitle(f"Figure 7: Residual diagnostics\\nStrategy: {STRATEGY_LABEL}", y=1.02, fontsize=12)
fig.tight_layout(); plt.show()

---
## S5. Summary for Strategy: `tit_for_tat_matcher`

Run this cell to print the consolidated findings.

In [ ]:
print("=" * 70)
print(f"  SUMMARY: {STRATEGY_LABEL}")
print("=" * 70)
print(f"  Data: {len(trans)} transitions from {len(runs_all)} runs")
print(f"  Products: {sorted(set(t['product'] for t in trans))}")
print(f"  Buyer types: {sorted(set(t['buyer_type'] for t in trans))}")
print()

# Best model info
X_base = np.column_stack([np.ones_like(fm), fm, fs])
b, rmse, r2, _, se = ols_fit(X_base, fy)
print(f"  Fractional baseline model:")
print(f"    frac(s_t) = {b[0]:+.3f} + {b[1]:+.3f}*frac(m_t) + {b[2]:+.3f}*frac(s_prev)")
print(f"    RMSE = {rmse:.4f} (fractional)  R2 = {r2:.4f}")
print()

# Regime counts
r0 = (m < v).sum()
r1 = ((m >= v) & (m <= s)).sum()
r2_count = (m > s).sum()
print(f"  Regime distribution:")
print(f"    m_t < v (below floor):      {r0:>5} ({r0/len(m)*100:.1f}%)")
print(f"    v <= m_t <= s_prev (zone):  {r1:>5} ({r1/len(m)*100:.1f}%)")
print(f"    m_t > s_prev (above):       {r2_count:>5} ({r2_count/len(m)*100:.1f}%)")
